# COEN803 Solo Assignment UEIR-Gated RAG Pipeline

**Student:** Abdulhameed Yunusa  
**Reg. No:** P24EGCP9020  
**Course:** COEN803 Advanced Programming, ABU Zaria (2025/2026)  
**Issued by:** Prof. T. H. Sikiru

---

## Application Domain: NLP / RAG

This notebook implements a **UEIR-Gated Retrieval-Augmented Generation (RAG) Pipeline**.  
Each document chunk is assigned a UEIR trace step. If that step's conductance Φ falls in the forbidden band `(0.5, 0.6]`, the chunk is **excluded** from the LLM context window — a real functional gate, not a visual label.

Three UEIR metrics are reported throughout: `phi_eff`, `stability_score`, `forbidden_events`.

## 1. Imports & Setup

In [1]:
import json
import math
import numpy as np
import anthropic
from typing import Dict, Any

from ueir_v3_5 import (
    recursive_ueir_step, UEIR_Orchestrator,
    UEIR_BlackHole, UEIR_Cosmology,
    CATEGORIES, K_SEED, N_SCREEN,
    assign_cat, get_prime, compute_n,
    is_forbidden, compute_conductance,
)

YOUR_NAME   = "Abdulhameed Yunusa"
YOUR_REG_NO = "P24EGCP9020"

print(f"UEIR v3.5 loaded. K_SEED={K_SEED}, N_SCREEN={N_SCREEN}")

UEIR v3.5 loaded. K_SEED=181, N_SCREEN=362


## 2. LLM Setup (Anthropic Claude Haiku)

The `call_llm()` function calls the Anthropic API. It reads `ANTHROPIC_API_KEY` from the environment.

In [2]:
def call_llm(system: str, user: str) -> str:
    """Calls Anthropic Claude Haiku. Reads ANTHROPIC_API_KEY from environment."""
    client = anthropic.Anthropic()
    msg = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        system=system,
        messages=[{"role": "user", "content": user}],
    )
    return msg.content[0].text

print("call_llm() ready.")

call_llm() ready.


## 3. Mandatory UEIR v3.5 Six-Point Scorecard

All 6 checks must pass before any application code runs.

| # | Check | Requirement |
|---|---|---|
| 1 | Φ ∈ (0,1) at every step | n > k always |
| 2 | cat ∈ [0,4] — no fallback | `assign_cat()` capped |
| 3 | Forbidden band triggered | ≥1 event |
| 4 | \|final − 0.5\| < 0.05 | Attractor convergence |
| 5 | Cat 4: 64 primes via sieve | 293 ≤ p ≤ 691 |
| 6 | k=181 structurally reached | At depth 17 |

In [3]:
def run_scorecard() -> bool:
    print("=" * 60)
    print(f"COEN803 Solo Assignment — {YOUR_NAME} ({YOUR_REG_NO})")
    print("UEIR v3.5 Scorecard")
    print("=" * 60)

    orch   = UEIR_Orchestrator(max_depth=24)
    result = orch.run(initial_state=0.5)
    trace  = orch.trace

    checks = {
        1: ("Phi in (0,1) at every step",
            all(0 < t["phi"] < 1 for t in trace),
            f"[min={min(t['phi'] for t in trace):.4f}, max={max(t['phi'] for t in trace):.4f}]"),
        2: ("cat in [0,4] — no fallback",
            all(0 <= t["cat"] <= 4 for t in trace),
            f"[cats: {sorted(set(t['cat'] for t in trace))}]"),
        3: ("Forbidden band triggered",
            result["forbidden_events"] > 0,
            f"{result['forbidden_events']} events"),
        4: ("|final - 0.5| < 0.05",
            abs(orch.final_state - 0.5) < 0.05,
            f"[distance={abs(orch.final_state-0.5):.6f}]"),
        5: ("Cat 4: 64 primes via sieve",
            len(CATEGORIES[4]) == 64,
            f"[{CATEGORIES[4][0]}..{CATEGORIES[4][-1]}]"),
        6: ("k=181 structurally reached",
            len(result['k181_at_depths']) > 0,
            f"at depth(s) {result['k181_at_depths']}"),
    }

    all_pass = True
    for i, (label, ok, detail) in checks.items():
        print(f"  {'OK' if ok else 'FAIL'} {i}. {label:<42s} {detail}")
        if not ok:
            all_pass = False

    print()
    print("  ALL CHECKS PASS — proceed to your application." if all_pass
          else "  ONE OR MORE CHECKS FAILED.")
    print("=" * 60)
    return all_pass

assert run_scorecard(), "Scorecard failed — fix before proceeding."

COEN803 Solo Assignment — Abdulhameed Yunusa (P24EGCP9020)
UEIR v3.5 Scorecard
  OK 1. Phi in (0,1) at every step                 [min=0.4805, max=0.5203]
  OK 2. cat in [0,4] — no fallback                 [cats: [0, 1, 2, 3, 4]]
  OK 3. Forbidden band triggered                   10 events
  OK 4. |final - 0.5| < 0.05                       [distance=0.004232]
  OK 5. Cat 4: 64 primes via sieve                 [293..691]
  OK 6. k=181 structurally reached                 at depth(s) [17]

  ALL CHECKS PASS — proceed to your application.


## 4. UEIR Orchestrator — Core Metrics

Run the UEIR v3.5 orchestrator and extract the three required metrics.

In [4]:
orch   = UEIR_Orchestrator(max_depth=24)
result = orch.run(initial_state=0.5, document_context=f"rag_nlp_{YOUR_REG_NO}")

phi_eff   = result["effective_conductance"]
stability = result["stability_score"]
n_forb    = result["forbidden_events"]
trace     = result["trace"]

print(f"phi_eff          : {phi_eff}")
print(f"stability_score  : {stability}")
print(f"forbidden_events : {n_forb}")
print(f"k=181 at depths  : {result['k181_at_depths']}")
print(f"phi range        : {result['phi_range']}")
print(f"trace steps      : {len(trace)}")

phi_eff          : 0.48899
stability_score  : 0.991537
forbidden_events : 10
k=181 at depths  : [17]
phi range        : (0.4805, 0.5203)
trace steps      : 25


## 5. UEIR Trace : Phi per Depth

Visualise the conductance Φ across all 25 depth steps, highlighting forbidden-band events.

In [5]:
print(f"{'Depth':>6} {'Cat':>4} {'k':>5} {'n':>6} {'Phi':>8} {'Forbidden':>10} {'Decision':>10}")
print("-" * 60)
for t in trace:
    decision = "REJECT" if t["forbidden"] else "approve"
    print(f"{t['depth']:>6} {t['cat']:>4} {t['k']:>5} {t['n']:>6} "
          f"{t['phi']:>8.4f} {'YES' if t['forbidden'] else 'no':>10} {decision:>10}")

 Depth  Cat     k      n      Phi  Forbidden   Decision
------------------------------------------------------------
     0    0     3      6   0.5000         no    approve
     1    1    13     27   0.4815         no    approve
     2    1    17     35   0.4857         no    approve
     3    2    43     89   0.4831         no    approve
     4    2    47     95   0.4947         no    approve
     5    3   113    223   0.5067        YES     REJECT
     6    3   127    245   0.5184        YES     REJECT
     7    3   131    252   0.5198        YES     REJECT
     8    3   137    267   0.5131        YES     REJECT
     9    3   139    278   0.5000         no    approve
    10    3   149    306   0.4869         no    approve
    11    3   151    314   0.4809         no    approve
    12    3   157    325   0.4831         no    approve
    13    3   163    330   0.4939         no    approve
    14    3   167    329   0.5076        YES     REJECT
    15    3   173    334   0.5180        YE

## 6. Document Corpus & UEIR Forbidden-Band Gate

Each of the 8 NLP/RAG chunks is assigned to a UEIR trace step (by `chunk_id % len(trace)`).  
If that step's Φ is in the forbidden band `(0.5, 0.6]` -> chunk is **REJECTED** from the LLM context.  

In [6]:
corpus = [
    {"id": 0, "text": "Retrieval-Augmented Generation (RAG) combines dense retrieval with generative models to produce grounded, citation-backed responses."},
    {"id": 1, "text": "Vector embeddings represent the semantic meaning of text in high-dimensional space, enabling similarity search via cosine or dot-product distance."},
    {"id": 2, "text": "BM25 is a probabilistic sparse retrieval method based on term frequency (TF) and inverse document frequency (IDF), effective for keyword-heavy queries."},
    {"id": 3, "text": "Chunking strategies for RAG include fixed-size windows, sentence-level splits, and semantic boundary detection to balance recall and context coherence."},
    {"id": 4, "text": "Re-ranking with cross-encoders improves retrieval precision by scoring query-chunk pairs jointly rather than independently."},
    {"id": 5, "text": "Hybrid retrieval fuses dense and sparse scores via Reciprocal Rank Fusion (RRF) to capture both semantic and lexical relevance signals."},
    {"id": 6, "text": "Hallucination in LLMs occurs when the model generates plausible but factually incorrect content not grounded in the retrieved context."},
    {"id": 7, "text": "UEIR conductance Phi = k/n measures the stability of information flow across recursive depth layers; the attractor at Phi=0.5 signals balanced retrieval confidence."},
]

approved     = []
rejected     = []
chunk_report = []

print(f"{'ID':>3} {'Depth':>6} {'Phi':>8} {'Forbidden':>10} {'Decision':>10}  Text excerpt")
print("-" * 90)

for chunk in corpus:
    step      = trace[chunk["id"] % len(trace)]
    phi_chunk = step["phi"]
    forbidden = step["forbidden"]
    decision  = "REJECT" if forbidden else "APPROVE"

    chunk_report.append({
        "chunk_id":       chunk["id"],
        "ueir_depth":     step["depth"],
        "phi":            phi_chunk,
        "forbidden_gate": forbidden,
        "decision":       decision,
        "text_excerpt":   chunk["text"][:60] + "...",
    })

    print(f"{chunk['id']:>3} {step['depth']:>6} {phi_chunk:>8.4f} {'YES' if forbidden else 'no':>10} "
          f"{decision:>10}  {chunk['text'][:55]}...")

    if forbidden:
        rejected.append(chunk)
    else:
        approved.append(chunk)

print()
print(f"Approved: {len(approved)} / {len(corpus)}")
print(f"Rejected: {len(rejected)} / {len(corpus)}")

 ID  Depth      Phi  Forbidden   Decision  Text excerpt
------------------------------------------------------------------------------------------
  0      0   0.5000         no    APPROVE  Retrieval-Augmented Generation (RAG) combines dense ret...
  1      1   0.4815         no    APPROVE  Vector embeddings represent the semantic meaning of tex...
  2      2   0.4857         no    APPROVE  BM25 is a probabilistic sparse retrieval method based o...
  3      3   0.4831         no    APPROVE  Chunking strategies for RAG include fixed-size windows,...
  4      4   0.4947         no    APPROVE  Re-ranking with cross-encoders improves retrieval preci...
  5      5   0.5067        YES     REJECT  Hybrid retrieval fuses dense and sparse scores via Reci...
  6      6   0.5184        YES     REJECT  Hallucination in LLMs occurs when the model generates p...
  7      7   0.5198        YES     REJECT  UEIR conductance Phi = k/n measures the stability of in...

Approved: 5 / 8
Rejected: 3 / 8


## 7. LLM Call

Only UEIR-approved chunks are passed to the LLM as context.

**System prompt** and **user prompt** are documented verbatim below as required by the assignment.

In [8]:
query = "Explain how retrieval quality is measured and improved in RAG pipelines."

system_prompt = (
    "You are an expert in Natural Language Processing and Retrieval-Augmented Generation (RAG). "
    "Answer the user's question using ONLY the provided document excerpts. "
    "If the excerpts do not contain sufficient information to answer fully, say so explicitly. "
    "Do not hallucinate or add information not present in the excerpts."
)

# --- Verbatim User Prompt ---
context_str = "\n".join(
    f"[Chunk {c['id']} | UEIR-APPROVED] {c['text']}" for c in approved
)
user_prompt = (
    f"Document excerpts approved by UEIR forbidden-band gate "
    f"(phi NOT in forbidden band (0.5, 0.6]):\n\n"
    f"{context_str}\n\n"
    f"Question: {query}"
)

print("=== SYSTEM PROMPT ===")
print(system_prompt)
print()
print("=== USER PROMPT ===")
print(user_prompt)

=== SYSTEM PROMPT ===
You are an expert in Natural Language Processing and Retrieval-Augmented Generation (RAG). Answer the user's question using ONLY the provided document excerpts. If the excerpts do not contain sufficient information to answer fully, say so explicitly. Do not hallucinate or add information not present in the excerpts.

=== USER PROMPT ===
Document excerpts approved by UEIR forbidden-band gate (phi NOT in forbidden band (0.5, 0.6]):

[Chunk 0 | UEIR-APPROVED] Retrieval-Augmented Generation (RAG) combines dense retrieval with generative models to produce grounded, citation-backed responses.
[Chunk 1 | UEIR-APPROVED] Vector embeddings represent the semantic meaning of text in high-dimensional space, enabling similarity search via cosine or dot-product distance.
[Chunk 2 | UEIR-APPROVED] BM25 is a probabilistic sparse retrieval method based on term frequency (TF) and inverse document frequency (IDF), effective for keyword-heavy queries.
[Chunk 3 | UEIR-APPROVED] Chunkin

In [9]:
# Call the LLM — only approved chunks reach Claude
llm_response = call_llm(system_prompt, user_prompt)

print("=== LLM RESPONSE (Claude Haiku) ===")
print(llm_response)

=== LLM RESPONSE (Claude Haiku) ===
# Retrieval Quality in RAG Pipelines

Based on the provided excerpts, I can address how retrieval quality is **improved** in RAG pipelines, though the documents do not explicitly discuss measurement metrics.

## Improvement Strategies

**1. Retrieval Method Selection**
The excerpts describe two complementary approaches:
- **Vector embeddings**: Enable semantic similarity search through high-dimensional space representations using cosine or dot-product distance
- **BM25**: A probabilistic sparse retrieval method using term frequency (TF) and inverse document frequency (IDF), particularly effective for keyword-heavy queries

**2. Chunking Strategy Optimization**
The excerpts note that chunking strategies—including fixed-size windows, sentence-level splits, and semantic boundary detection—can be tuned to "balance recall and context coherence."

**3. Re-ranking with Cross-Encoders**
Re-ranking improves retrieval precision by scoring query-chunk pairs **j

## 8. Break Report

Two attempts to break UEIR v3.5, with documented outcomes and proposed fixes.

### Attempt 1 — Truncate `max_depth` to 4

**Rationale:** Forbidden events first occur at depth 5; k=181 emerges at depth 17.  
Stopping at depth 4 should break checks 3 and 6.

In [10]:
orch1   = UEIR_Orchestrator(max_depth=4)
result1 = orch1.run(initial_state=0.5)

c3 = result1["forbidden_events"] > 0
c6 = len(result1["k181_at_depths"]) > 0

print(f"max_depth=4 results:")
print(f"  forbidden_events : {result1['forbidden_events']}")
print(f"  k181_at_depths   : {result1['k181_at_depths']}")
print(f"  Check 3 (forbidden band triggered) : {'PASS' if c3 else 'FAIL'}")
print(f"  Check 6 (k=181 structurally reached): {'PASS' if c6 else 'FAIL'}")
print()
print("Proposed fix: assert max_depth >= 24 in run_scorecard() before instantiating the orchestrator.")

max_depth=4 results:
  forbidden_events : 0
  k181_at_depths   : []
  Check 3 (forbidden band triggered) : FAIL
  Check 6 (k=181 structurally reached): FAIL

Proposed fix: assert max_depth >= 24 in run_scorecard() before instantiating the orchestrator.


### Attempt 2 — Unphysical `initial_state = 10.0`

**Rationale:** Test whether the attractor pull converges from a state far outside `(0, 1)`,  
and whether phi bounds (check 1) are violated. Since Φ = k/n is independent of `initial_state`,  
phi bounds should remain intact.

In [11]:
orch2   = UEIR_Orchestrator(max_depth=24)
result2 = orch2.run(initial_state=10.0)

final2  = orch2.final_state
dist2   = abs(final2 - 0.5)
phi_ok  = all(0 < t["phi"] < 1 for t in orch2.trace)
c4      = dist2 < 0.05

print(f"initial_state=10.0 results:")
print(f"  final_state          : {round(final2, 6)}")
print(f"  |final - 0.5|        : {round(dist2, 6)}")
print(f"  phi bounds preserved : {phi_ok}  (phi=k/n, independent of initial_state)")
print(f"  Check 4 (attractor)  : {'PASS' if c4 else 'FAIL'}")
print()
print("Proposed fix: add 'assert 0 < initial_state < 1' at the top of UEIR_Orchestrator.run().")

initial_state=10.0 results:
  final_state          : 6.073129
  |final - 0.5|        : 5.573129
  phi bounds preserved : True  (phi=k/n, independent of initial_state)
  Check 4 (attractor)  : FAIL

Proposed fix: add 'assert 0 < initial_state < 1' at the top of UEIR_Orchestrator.run().


## 9. Final Summary

In [12]:
summary = {
    "student":           YOUR_NAME,
    "reg_no":            YOUR_REG_NO,
    "application":       "UEIR-Gated RAG Pipeline",
    "domain":            "NLP/RAG",
    "phi_eff":           phi_eff,
    "stability_score":   stability,
    "forbidden_events":  n_forb,
    "corpus_size":       len(corpus),
    "approved_chunks":   len(approved),
    "rejected_chunks":   len(rejected),
    "llm_model":         "claude-haiku-4-5-20251001",
    "scorecard":         "ALL 6 CHECKS PASS",
}

print("Final Summary")
print("-" * 40)
for k, v in summary.items():
    print(f"  {k:<22}: {v}")
print()

Final Summary
----------------------------------------
  student               : Abdulhameed Yunusa
  reg_no                : P24EGCP9020
  application           : UEIR-Gated RAG Pipeline
  domain                : NLP/RAG
  phi_eff               : 0.48899
  stability_score       : 0.991537
  forbidden_events      : 10
  corpus_size           : 8
  approved_chunks       : 5
  rejected_chunks       : 3
  llm_model             : claude-haiku-4-5-20251001
  scorecard             : ALL 6 CHECKS PASS

